# Data Cleaning 

## 1. Imports & load raw data

In [28]:
import pandas as pd
import numpy as np

INPUT_FILE = "CEAS_08.csv"   
df = pd.read_csv("C:\\Users\\PIXEL-PC\\Desktop\\SIC_Capstone\\phishing-email-detector\\data\\raw\\CEAS_08.csv")
print("Starting shape:", df.shape)

Starting shape: (39154, 7)


## 2. Handle missing values

In [29]:
df["receiver"] = df["receiver"].fillna("")
df["subject"] = df["subject"].fillna("")

print("Missing values remaining:")
print(df.isnull().sum())

Missing values remaining:
sender      0
receiver    0
date        0
subject     0
body        0
label       0
urls        0
dtype: int64


## 3. Body length (outliers)
I've Checked the 3 longest emails from EDA (up to 144k chars) and they're all real, legit emails. So decided to leave them as they are

I kept the `body_length` column in the final dataset though it might be useful for ML model

In [30]:
df["body_length"] = df["body"].astype(str).apply(len)
print(df["body_length"].describe())

count     39154.000000
mean       1571.079813
std        3615.810576
min          14.000000
25%         224.000000
50%         570.000000
75%        1643.000000
max      143996.000000
Name: body_length, dtype: float64


## 4. Date column
While doing EDA we found dates that don't make sense (like year 2100) or fall outside a realistic range for this dataset (before 1990 or after 2026) 26 rows total.

Decision: not dropping these rows, since the email itself is still valid even if the date field is messed up. Instead, parsed the date into a proper datetime column, and treated anything outside the realistic 1990–2026 range as missing (`NaT`) rather than a valid date. Keeping this simple no extra "suspicious date" flag column

In [31]:
df["date_parsed"] = pd.to_datetime(df["date"], errors="coerce", utc=True)

impossible = (df["date_parsed"] < "1990-01-01") | (df["date_parsed"] > "2026-12-31")
df.loc[impossible, "date_parsed"] = pd.NaT

print("impossible dates:", df["date_parsed"].isna().sum())

impossible dates: 26


In [32]:
print(df["date_parsed"].describe())
print(df["date_parsed"].dt.year.value_counts().sort_index())

count                                  39128
mean     2008-07-31 04:47:17.799938816+00:00
min                1998-03-02 14:32:36+00:00
25%      2008-08-06 06:58:37.750000128+00:00
50%         2008-08-07 01:56:36.500000+00:00
75%      2008-08-07 19:29:38.249999872+00:00
max                2020-06-19 22:30:58+00:00
Name: date_parsed, dtype: object
date_parsed
1998.0        2
1999.0        1
2000.0       25
2001.0       13
2002.0       33
2003.0       16
2004.0       10
2005.0       18
2006.0        7
2007.0       37
2008.0    38947
2009.0       15
2010.0        1
2012.0        1
2020.0        2
Name: count, dtype: int64


## 5. Label column check

In [33]:
print(df["label"].value_counts())
print(df["label"].dtype)

label
1    21842
0    17312
Name: count, dtype: int64
int64


## 6. Sender / Subject quality
In EDA we found 955 rows with a malformed sender, 2,762 rows with leading/trailing whitespace in subject, and 151 rows where sender equals receiver.

Decision: trimming whitespace from sender/receiver/subject since it's a safe, easy fix. Not touching the malformed senders or sender==receiver rows they're unusual but still valid data, just flagging



In [34]:
df["sender"] = df["sender"].str.strip()
df["receiver"] = df["receiver"].str.strip()
df["subject"] = df["subject"].str.strip()

has_ws = df["subject"].apply(lambda x: x != x.strip())
print("Subjects with whitespace remaining:", has_ws.sum())

bad_senders = (~df["sender"].str.contains("@", na=False)).sum()
same_sender_receiver = (df["sender"] == df["receiver"]).sum()
print("Malformed senders (no @):", bad_senders)
print("Sender == Receiver:", same_sender_receiver)

Subjects with whitespace remaining: 0
Malformed senders (no @): 955
Sender == Receiver: 151


## 7. Save cleaned dataset

In [35]:
OUTPUT_FILE = r"C:\Users\PIXEL-PC\Desktop\SIC_Capstone\phishing-email-detector\data\processed\cleaned_dataset.csv"
df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {OUTPUT_FILE} — shape {df.shape}")
df.head()


Saved C:\Users\PIXEL-PC\Desktop\SIC_Capstone\phishing-email-detector\data\processed\cleaned_dataset.csv — shape (39154, 9)


,sender,receiver,date,subject,body,label,urls,body_length,date_parsed
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1,273,2008-08-05 23:31:02+00:00
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1,82,2008-08-05 23:31:03+00:00
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1,3918,2008-08-06 08:28:00+00:00
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0,1,24418,2008-08-05 23:31:20+00:00
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1,1,175,2008-08-05 23:31:21+00:00


In [36]:
import os
print(os.path.exists(r"C:\Users\PIXEL-PC\Desktop\SIC_Capstone\phishing-email-detector\data\processed\cleaned_dataset.csv"))
print(df.shape)
print(df.columns.tolist())

True
(39154, 9)
['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls', 'body_length', 'date_parsed']


## Notes for the team
- Missing values: `receiver` and `subject` filled with empty string, no rows dropped
- No duplicate rows found
- `body_length` column added (may be useful for ML model)
- `body` text left completely untouched Buthaina is handling NLP-specific text cleaning on her end
- `date_parsed` column added (proper datetime); dates outside 1990–2026 treated as missing (NaT)
- `sender`, `receiver`, `subject` had whitespace trimmed
- 955 rows have malformed sender addresses (no real email inside <>) flagging for whoever does sender/URL analysis
- 151 rows have sender == receiver "unusual but left as-is no changes made"
- Final dataset: same row count as raw (39,154), no rows dropped